# LLM-based Toxicity Moderation: SOTA models (batch API)

Few-shot conversation-level toxicity classification with top-tier LLMs, submitted through each provider's **batch API** (50% off the standard rate, up to 24h turnaround).

**Models:** Claude Opus 5, GPT-5.6 Sol, Gemini 3.7 Flash.
**Datasets:** Pleroma and Mastodon.
**Prompt variants:** base, and a threshold-rules variant that states the conversation-level labelling rule.
**Seeds:** {1, 42, 999}.

All 72 jobs (3 providers x 2 datasets x 3 seeds x 2 prompt variants x 2 test sets) are submitted and collected by a single call to `run_all_hardened` in Section 7. Progress is persisted to `run_manifest.json`, so the run is resumable: re-running the same call skips completed jobs and never resubmits them.

**Inputs:** this notebook reuses the test sets and few-shot prompts built by the lightweight-model notebook. It reads that notebook's per-seed `local_predictions.csv`, `shared_global_predictions.csv`, and `few_shot_examples.csv`, so the two model tiers are evaluated on exactly the same conversations.

**Conversation filter:** only conversations with at least `MIN_POSTS_PER_CONVERSATION` posts are used, matching the conversation-length filter applied in the federated GNN experiments.

## 1. Install dependencies

In [ ]:
!pip -q install requests pandas numpy scikit-learn

## 2. Setup and configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, json, hashlib, requests
from pathlib import Path
import pandas as pd

# OUTPUT_BASE must contain one subfolder per dataset each holding the per-seed outputs written by the lightweight-model notebook.
OUTPUT_BASE = Path("<PATH_TO>/LLM_Experiments")
BATCH_DIR = Path("<PATH_TO>/LLM_Experiments/sota_batches")
BATCH_DIR.mkdir(parents=True, exist_ok=True)

# folder name, inside each dataset directory holding the run whose test sets and few-shot prompts are reused here
USAGE_LOG = BATCH_DIR / "usage_log.csv"
SOURCE_MODEL_FOLDER = "<LIGHTWEIGHT_RUN_FOLDER>" # e.g. "claude_haiku_4.5_results"

# get API keys from Colab
try:
    from google.colab import userdata
    def _secret(name):
        try:
            return userdata.get(name) or ""
        except Exception:
            return os.environ.get(name, "")
except Exception:
    def _secret(name):
        return os.environ.get(name, "")

API_KEYS = {
    "anthropic": _secret("ANTHROPIC_API_KEY"),
    "openai": _secret("OPENAI_API_KEY"),
    "gemini": _secret("GEMINI_API_KEY"),
}

print("Keys loaded:", {k: bool(v) for k, v in API_KEYS.items()})
print("Output base:", OUTPUT_BASE)
print("Batch dir:", BATCH_DIR)
print("Usage log:", USAGE_LOG)

## 3. Token-usage capture

Shared schema written to `usage_log.csv` during the run, consumed by the cost cell at the end. This cell is identical in both notebooks.

In [ ]:
import pandas as pd
from pathlib import Path

USAGE_KEYS = ["provider", "model", "dataset", "seed", "variant", "eval_type"]
USAGE_SUMS = ["n_requests", "input_tokens", "cached_read_tokens", "cache_write_tokens", "output_tokens", "reasoning_tokens"]

def normalize_usage(provider, usage):
    u = usage or {}
    if provider == "anthropic":
        read = u.get("cache_read_input_tokens", 0) or 0
        write = u.get("cache_creation_input_tokens", 0) or 0
        return dict(input_tokens=(u.get("input_tokens", 0) or 0) + read + write,
                    cached_read_tokens=read, cache_write_tokens=write, output_tokens=u.get("output_tokens", 0) or 0, reasoning_tokens=0)
    if provider == "openai":
        return dict(input_tokens=u.get("prompt_tokens", 0) or 0,
                    cached_read_tokens=(u.get("prompt_tokens_details") or {}).get("cached_tokens", 0) or 0,
                    cache_write_tokens=0, output_tokens=u.get("completion_tokens", 0) or 0,
                    reasoning_tokens=(u.get("completion_tokens_details") or {}).get("reasoning_tokens", 0) or 0)
    if provider == "gemini":
        thoughts = u.get("thoughtsTokenCount", 0) or 0
        return dict(input_tokens=u.get("promptTokenCount", 0) or 0,
                    cached_read_tokens=u.get("cachedContentTokenCount", 0) or 0,
                    cache_write_tokens=0, output_tokens=(u.get("candidatesTokenCount", 0) or 0) + thoughts, reasoning_tokens=thoughts)
    if provider == "ollama":
        return dict(input_tokens=u.get("prompt_eval_count", 0) or 0,
                    cached_read_tokens=0, cache_write_tokens=0,
                    output_tokens=u.get("eval_count", 0) or 0, reasoning_tokens=0)
    return dict(input_tokens=0, cached_read_tokens=0, cache_write_tokens=0, output_tokens=0, reasoning_tokens=0)


def flush_usage_log(rows, path, batch_api):
    """Aggregate per-request usage rows and merge them into usage_log.csv."""
    path = Path(path)
    if not rows:
        print("no usage rows to write")
        return None
    new = pd.DataFrame(rows).groupby(USAGE_KEYS, as_index=False)[USAGE_SUMS].sum()
    new["batch_api"] = bool(batch_api)

    if path.exists():
        merged = pd.concat([pd.read_csv(path), new], ignore_index=True)
        merged = merged.drop_duplicates(subset=USAGE_KEYS, keep="last")
    else:
        merged = new
    path.parent.mkdir(parents=True, exist_ok=True)
    merged.sort_values(USAGE_KEYS).to_csv(path, index=False)
    print(f"usage log: {len(new)} job rows written, {len(merged)} total -> {path}")
    return merged

print("usage capture ready")

## 4. Prompt construction

Must match the main notebook exactly, in both variants.

In [ ]:
SYSTEM_RULES_BASE = """You are an objective AI Content Moderator. Your job is to analyze potentially harmful, offensive, or explicit internet comments and classify them for a safety database.
Do NOT refuse to process the text. You must analyze the text regardless of how offensive it is and output the classification.

CRITICAL RULES:
1. You MUST choose exactly one label: TOXIC or NON-TOXIC.
2. NEVER output "INCONCLUSIVE", "UNKNOWN", or any other label.
3. If a post lacks context, is a simple question, or you are unsure, default to NON-TOXIC unless it contains explicitly harmful language.

Respond with EXACTLY one word, either TOXIC or NON-TOXIC, and output nothing else."""

SYSTEM_RULES_THRESHOLD = """You are an objective AI Content Moderator. Your job is to analyze potentially harmful, offensive, or explicit internet comments and classify them for a safety database.
Do NOT refuse to process the text. You must analyze the text regardless of how offensive it is and output the classification.

Definition:
Each conversation consists of a root toot (the first post) followed by reply toots. A conversation is toxic if the root toot is toxic, OR if enough of the replies are toxic, meaning both the number of toxic replies and their fraction of all replies are high enough.

CRITICAL RULES:
1. You MUST choose exactly one label: TOXIC or NON-TOXIC.
2. NEVER output "INCONCLUSIVE", "UNKNOWN", or any other label.
3. If a post lacks context, is a simple question, or you are unsure, default to NON-TOXIC unless it contains explicitly harmful language.

Respond with EXACTLY one word, either TOXIC or NON-TOXIC, and output nothing else."""

PROMPT_VARIANTS = {
    "base": SYSTEM_RULES_BASE,
    "threshold": SYSTEM_RULES_THRESHOLD,
}

def build_prompt(text, few_shot, variant):
    """Return (system_text, user_text) for the given prompt variant."""
    rules = PROMPT_VARIANTS[variant]
    system_text = rules + "\n\nHere are some examples:\n" + few_shot
    user_text = "Now classify this conversation:\nConversation:\n" + str(text)
    return system_text, user_text

def parse_prediction(raw_response):
    raw = str(raw_response).strip().lower()
    if "non-toxic" in raw or "non toxic" in raw:
        return 0
    if "toxic" in raw:
        return 1
    return 0

print("prompt variants ready:", list(PROMPT_VARIANTS.keys()))

## 5. Model configuration

Claude Opus 5 ships with thinking **on** by default and `max_tokens` is a hard cap on thinking + visible
output, so thinking is disabled explicitly here and `max_tokens` is raised. Disabling thinking is only
permitted at effort `high` or below (`high` is the default, so nothing else is needed).

In [ ]:
MODEL_CONFIGS = {
    "anthropic": {
        "model": "claude-opus-5",
        "max_tokens": 256,
        "thinking": {"type": "disabled"},
    },
    "openai": {
        "model": "gpt-5.6-sol",
        "max_completion_tokens": 128,
        "reasoning_effort": "low",
    },
    "gemini": {
        "model": "gemini-3.7-flash",
        "max_output_tokens": 128,
        "thinking_level": "low",
    },
}
print(MODEL_CONFIGS)

## 6. Loading test rows + building the JSONL requests

Each provider uses a different batch JSONL schema, so one file per provider. Batch APIs restrict `custom_id`/`key` to ~64 safe characters, so ids are hashed and the real metadata is kept in a side-car `id_map`.

In [ ]:
SOURCE_MODEL_FOLDER = "claude_haiku_4.5_results"

def load_job_rows(output_base: Path, dataset: str, seed: int, eval_type: str):
    """
    dataset: "pleroma" or "mastodon" - subfolder name under output_base
    eval_type: "local" or "shared_global"
    Returns a DataFrame with: model_instance_id, conversation_id, ground_truth, text, few_shot
    """
    seed_dir = output_base / dataset / SOURCE_MODEL_FOLDER / f"rs{seed}"
    pred_file = seed_dir / f"{'local' if eval_type == 'local' else 'shared_global'}_predictions.csv"
    fewshot_file = output_base / dataset / "few_shot_examples" / f"rs{seed}" / "few_shot_examples.csv"

    if not pred_file.exists():
        raise FileNotFoundError(f"Missing {pred_file}")
    if not fewshot_file.exists():
        raise FileNotFoundError(
            f"Missing {fewshot_file} - add the few-shot export cell (Section 0) to the main "
            f"notebook and re-run it for this seed."
        )

    preds = pd.read_csv(pred_file)[["model_instance_id", "conversation_id", "ground_truth", "text"]]
    fewshot = pd.read_csv(fewshot_file)[["model_instance_id", "few_shot"]]

    merged = preds.merge(fewshot, on="model_instance_id", how="left", validate="many_to_one")
    missing = merged["few_shot"].isna().sum()
    if missing:
        raise ValueError(f"{missing} rows have no matching few-shot block - check model_instance_id join.")
    return merged

import hashlib

def make_custom_id(dataset, seed, variant, eval_type, model_instance_id, conversation_id):
    """Batch APIs (Anthropic/OpenAI/Gemini) restrict custom_id/key to something like 
    ^[a-zA-Z0-9_-]{1,64}$ - no pipes, 64 char cap. model_instance_id/conversation_id can be
    UUIDs, so packing everything into the id itself is both invalid and too long. Hash it down
    to a short safe id instead, and keep the real metadata in a side-car id_map (see
    build_id_map_entry / write_id_map / load_id_map below)."""
    original = f"{dataset}|{seed}|{variant}|{eval_type}|{model_instance_id}|{conversation_id}"
    short = hashlib.sha256(original.encode()).hexdigest()[:20]
    return f"{dataset}_{seed}_{short}"

def build_id_map_entry(dataset, seed, variant, eval_type, model_instance_id, conversation_id):
    return {
        "dataset": dataset,
        "random_state": int(seed),
        "prompt_variant": variant,
        "eval_type": eval_type,
        "model_instance_id": model_instance_id,
        "conversation_id": conversation_id,
    }

def write_id_map(id_map: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(id_map, f)
    return path

def load_id_map(path: Path):
    with open(path) as f:
        return json.load(f)

In [ ]:
def build_anthropic_jsonl_lines(rows_df, dataset, seed, variant, eval_type):
    cfg = MODEL_CONFIGS["anthropic"]
    lines = []
    id_map = {}
    for _, row in rows_df.iterrows():
        system_text, user_text = build_prompt(row["text"], row["few_shot"], variant)
        cid = make_custom_id(dataset, seed, variant, eval_type, row["model_instance_id"], row["conversation_id"])
        id_map[cid] = build_id_map_entry(dataset, seed, variant, eval_type, row["model_instance_id"], row["conversation_id"])

        params = {
            "model": cfg["model"],
            "max_tokens": cfg["max_tokens"],
            "system": [{"type": "text", "text": system_text, "cache_control": {"type": "ephemeral"}}],
            "messages": [{"role": "user", "content": user_text}],
        }
        if cfg.get("thinking"):
            params["thinking"] = cfg["thinking"]

        lines.append({"custom_id": cid, "params": params})
    return lines, id_map

def build_openai_jsonl_lines(rows_df, dataset, seed, variant, eval_type):
    cfg = MODEL_CONFIGS["openai"]
    lines = []
    id_map = {}
    for _, row in rows_df.iterrows():
        system_text, user_text = build_prompt(row["text"], row["few_shot"], variant)
        cid = make_custom_id(dataset, seed, variant, eval_type, row["model_instance_id"], row["conversation_id"])
        id_map[cid] = build_id_map_entry(dataset, seed, variant, eval_type, row["model_instance_id"], row["conversation_id"])
        body = {
            "model": cfg["model"],
            "messages": [
                {"role": "system", "content": system_text},
                {"role": "user", "content": user_text},
            ],
            "max_completion_tokens": cfg["max_completion_tokens"],
        }
        if cfg.get("reasoning_effort"):
            body["reasoning_effort"] = cfg["reasoning_effort"]

        lines.append({"custom_id": cid, "method": "POST", "url": "/v1/chat/completions", "body": body})
    return lines, id_map

def build_gemini_jsonl_lines(rows_df, dataset, seed, variant, eval_type):
    cfg = MODEL_CONFIGS["gemini"]
    lines = []
    id_map = {}
    for _, row in rows_df.iterrows():
        system_text, user_text = build_prompt(row["text"], row["few_shot"], variant)
        key = make_custom_id(dataset, seed, variant, eval_type, row["model_instance_id"], row["conversation_id"])
        id_map[key] = build_id_map_entry(dataset, seed, variant, eval_type, row["model_instance_id"], row["conversation_id"])
        gen_cfg = {
            "maxOutputTokens": cfg["max_output_tokens"],
            "responseMimeType": "text/x.enum",
            "responseSchema": {"type": "STRING", "enum": ["TOXIC", "NON-TOXIC"]},
        }
        if cfg.get("thinking_level"):
            gen_cfg["thinkingConfig"] = {"thinkingLevel": cfg["thinking_level"]}
        lines.append({
            "key": key,
            "request": {
                "systemInstruction": {"parts": [{"text": system_text}]},
                "contents": [{"parts": [{"text": user_text}]}],
                "generationConfig": gen_cfg,
            },
        })
    return lines, id_map

BUILDERS = {
    "anthropic": build_anthropic_jsonl_lines,
    "openai": build_openai_jsonl_lines,
    "gemini": build_gemini_jsonl_lines
}

def write_jsonl(lines, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for line in lines:
            f.write(json.dumps(line) + "\n")
    return path

def generate_batch_file(output_base: Path, batch_dir: Path, provider: str, dataset: str, seed: int, variant: str, eval_type: str):
    rows_df = load_job_rows(output_base, dataset, seed, eval_type)
    lines, id_map = BUILDERS[provider](rows_df, dataset, seed, variant, eval_type)
    fname = f"{provider}_{dataset}_rs{seed}_{variant}_{eval_type}.jsonl"
    path = write_jsonl(lines, batch_dir / provider / fname)
    idmap_path = write_id_map(id_map, batch_dir / provider / f"{fname}.idmap.json")
    print(f"wrote {len(lines)} requests -> {path} (id map -> {idmap_path})")
    return path, idmap_path

print("builders ready")

## 6b. Verify the inherited conversation-length filter

This notebook never reads the raw posts CSVs, so it cannot apply the filter itself. Every test row and few-shot block comes from the lightweight notebook, which keeps only conversations with at least `MIN_POSTS_PER_CONVERSATION` posts. This cell checks that the inherited files really do reflect that filter, and raises before any batch is submitted if they do not.

In [ ]:
import re

MIN_POSTS_PER_CONVERSATION = 10
POST_SEPARATOR = "\n---\n"
CLEANED_SEPARATOR = " --- "

POSTS_CSV_BY_DATASET = {
    "pleroma": "<PATH_TO>/pleroma_data/pleroma_data_after_filtering.csv",
    "mastodon": "<PATH_TO>/mastodon_data/mastodon_data_after_filtering.csv",
}

POSTS_CONV_COL = "conversation_id"

VERIFY_DATASETS = ("pleroma", "mastodon")
VERIFY_SEEDS = (1, 42, 999)
VERIFY_EVAL_TYPES = ("local", "shared_global")

_post_count_cache = {}

def _true_post_counts(dataset):
    """Posts per conversation from the raw posts CSV, or None if no path is configured."""
    path = POSTS_CSV_BY_DATASET.get(dataset)
    if not path or str(path).startswith("<PATH_TO>"):
        return None
    if dataset not in _post_count_cache:
        posts = pd.read_csv(path, usecols=[POSTS_CONV_COL])
        _post_count_cache[dataset] = posts.groupby(POSTS_CONV_COL).size()
    return _post_count_cache[dataset]

def count_posts_from_text(text, separator=POST_SEPARATOR):
    return str(text).count(separator) + 1

def _fewshot_example_post_counts(few_shot_block):
    """Inferred post count for each example inside one few-shot block."""
    examples = re.findall(r"Conversation:\n(.*)", str(few_shot_block))
    return [count_posts_from_text(ex, CLEANED_SEPARATOR) for ex in examples]

def verify_length_filter(output_base, datasets=VERIFY_DATASETS, seeds=VERIFY_SEEDS, eval_types=VERIFY_EVAL_TYPES, min_posts=MIN_POSTS_PER_CONVERSATION, strict=True):
    report_rows = []
    violations = []

    for dataset in datasets:
        true_counts = _true_post_counts(dataset)
        method = "exact" if true_counts is not None else "inferred"

        for seed in seeds:
            for eval_type in eval_types:
                try:
                    df = load_job_rows(output_base, dataset, seed, eval_type)
                except FileNotFoundError as exc:
                    print(f"SKIP  {dataset} rs{seed} {eval_type}: {exc}")
                    continue

                if true_counts is not None:
                    counts = df["conversation_id"].map(true_counts)
                    unmatched = int(counts.isna().sum())
                else:
                    counts = df["text"].map(count_posts_from_text)
                    unmatched = 0

                short_mask = counts.notna() & (counts < min_posts)
                rows_bad = int(short_mask.sum())
                convs_bad = int(df.loc[short_mask, "conversation_id"].nunique())
                counts = counts.dropna().astype(int)

                fewshot_counts = []
                for block in df["few_shot"].dropna().unique():
                    fewshot_counts.extend(_fewshot_example_post_counts(block))
                fewshot_bad = sum(1 for c in fewshot_counts if c < min_posts)

                report_rows.append({
                    "dataset": dataset,
                    "seed": seed,
                    "eval_type": eval_type,
                    "method": method,
                    "test_rows": len(df),
                    "unique_conversations": df["conversation_id"].nunique(),
                    "min_posts_seen": int(counts.min()) if len(counts) else -1,
                    "median_posts": float(counts.median()) if len(counts) else -1.0,
                    "rows_below_min": rows_bad,
                    "conversations_below_min": convs_bad,
                    "unmatched_in_posts_csv": unmatched,
                    "fewshot_examples": len(fewshot_counts),
                    "fewshot_below_min": fewshot_bad,
                })

                if rows_bad or fewshot_bad:
                    violations.append(
                        f"{dataset} rs{seed} {eval_type} [{method}]: {convs_bad} unique test "
                        f"conversations ({rows_bad} scored rows) and {fewshot_bad} few-shot "
                        f"examples have fewer than {min_posts} posts"
                    )
                if unmatched:
                    violations.append(
                        f"{dataset} rs{seed} {eval_type}: {unmatched} conversation_ids are "
                        f"absent from the posts CSV, so their length is unverified"
                    )

    if not report_rows:
        raise FileNotFoundError(
            "No source prediction files were found. Check OUTPUT_BASE and "
            f"SOURCE_MODEL_FOLDER (currently {SOURCE_MODEL_FOLDER!r})."
        )

    report = pd.DataFrame(report_rows)
    display(report)

    if violations:
        message = (
            f"Conversation-length filter (>= {min_posts} posts) is NOT satisfied by the "
            f"inherited source run in {len(violations)} of {len(report_rows)} job inputs:\n  "
            + "\n  ".join(violations)
        )
        if strict:
            raise AssertionError(message)
        print("WARNING:", message)
    else:
        methods = sorted(report["method"].unique())
        print(f"OK: all {len(report_rows)} job inputs satisfy >= {min_posts} posts per "
              f"conversation (method: {', '.join(methods)}; "
              f"minimum seen: {int(report['min_posts_seen'].min())}).")

    return report

length_filter_report = verify_length_filter(OUTPUT_BASE)

## 7. Submit / poll / download

**OpenAI specifics:** the input file is uploaded, waited on until `processed`, used to create the batch with retries, and then **deleted**. That last step matters: OpenAI copies the input at batch-creation time, so the upload is disposable, and leaving them behind fills the storage quota and makes later batches fail with `Cannot find file ...`.

In [ ]:
def _openai_upload_and_create(jsonl_path: Path, api_key: str):
    """Upload -> wait for 'processed' -> create batch (with retries) -> delete the upload."""
    with open(jsonl_path, "rb") as f:
        file_resp = requests.post(
            "https://api.openai.com/v1/files",
            headers={"Authorization": f"Bearer {api_key}"},
            files={"file": (jsonl_path.name, f, "application/jsonl")},
            data={"purpose": "batch"},
        )
    if not file_resp.ok:
        print(f"[openai] {file_resp.status_code} error body (file upload):", file_resp.text[:500])
    file_resp.raise_for_status()
    file_id = file_resp.json()["id"]

    status = None
    for _ in range(60):
        chk = requests.get(f"https://api.openai.com/v1/files/{file_id}", headers={"Authorization": f"Bearer {api_key}"})
        if chk.status_code == 200:
            status = chk.json().get("status")
            if status == "processed":
                break
            if status == "error":
                raise RuntimeError(f"file {file_id} failed processing: {chk.json()}")
        time.sleep(3)
    else:
        raise RuntimeError(f"file {file_id} stuck at status={status} after 180s")

    last = None
    batch_id = None
    for attempt in range(5):
        batch_resp = requests.post(
            "https://api.openai.com/v1/batches",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={"input_file_id": file_id, "endpoint": "/v1/chat/completions",
                  "completion_window": "24h"},
        )
        if batch_resp.status_code < 400:
            batch_id = batch_resp.json()["id"]
            break
        last = batch_resp.text[:400]
        time.sleep(5 * (attempt + 1))

    return batch_id

def submit_batch(provider: str, jsonl_path: Path, api_key: str):
    if provider == "anthropic":
        requests_list = [json.loads(l) for l in open(jsonl_path)]
        resp = requests.post(
            "https://api.anthropic.com/v1/messages/batches",
            headers={"x-api-key": api_key, "anthropic-version": "2023-06-01", "content-type": "application/json"},
            json={"requests": requests_list},
        )
        if not resp.ok:
            print(f"[anthropic] {resp.status_code} error body:", resp.text[:500])
        resp.raise_for_status()
        return resp.json()["id"]

    if provider == "openai":
        return _openai_upload_and_create(jsonl_path, api_key)

    if provider == "gemini":
        with open(jsonl_path, "rb") as f:
            file_resp = requests.post(
                f"https://generativelanguage.googleapis.com/upload/v1beta/files?key={api_key}",
                headers={"X-Goog-Upload-Protocol": "raw", "Content-Type": "application/jsonl"},
                data=f.read(),
            )
        if not file_resp.ok:
            print(f"[gemini] {file_resp.status_code} error body (file upload):", file_resp.text[:500])
        file_resp.raise_for_status()
        file_name = file_resp.json()["file"]["name"]
        model = MODEL_CONFIGS["gemini"]["model"]
        batch_resp = requests.post(
            f"https://generativelanguage.googleapis.com/v1beta/models/{model}:batchGenerateContent?key={api_key}",
            json={"batch": {"inputConfig": {"fileName": file_name}}},
        )
        if not batch_resp.ok:
            print(f"[gemini] {batch_resp.status_code} error body (batch create):", batch_resp.text[:500])
        batch_resp.raise_for_status()
        return batch_resp.json()["name"]

    raise ValueError(provider)

print("submit_batch ready")

In [ ]:
def poll_batch(provider: str, batch_id: str, api_key: str, interval=60, timeout=24 * 3600):
    start = time.time()
    while time.time() - start < timeout:
        if provider == "anthropic":
            r = requests.get(f"https://api.anthropic.com/v1/messages/batches/{batch_id}", headers={"x-api-key": api_key, "anthropic-version": "2023-06-01"})
            r.raise_for_status()
            status = r.json()["processing_status"]
            if status == "ended":
                return r.json()
        elif provider == "openai":
            r = requests.get(f"https://api.openai.com/v1/batches/{batch_id}", headers={"Authorization": f"Bearer {api_key}"})
            r.raise_for_status()
            status = r.json()["status"]
            if status in ("completed", "failed", "expired", "cancelled"):
                return r.json()
        elif provider == "gemini":
            r = requests.get(f"https://generativelanguage.googleapis.com/v1beta/{batch_id}?key={api_key}")
            r.raise_for_status()
            state = r.json().get("metadata", {}).get("state", "")
            if state in ("BATCH_STATE_SUCCEEDED", "BATCH_STATE_FAILED", "BATCH_STATE_CANCELLED"):
                return r.json()
        print(f"[{provider}] still processing, waiting {interval}s...")
        time.sleep(interval)
    raise TimeoutError(f"{provider} batch {batch_id} did not finish within {timeout}s")


def download_and_parse(provider: str, batch_info: dict, api_key: str, id_map: dict) -> pd.DataFrame:
    records = []

    if provider == "anthropic":
        results_url = batch_info["results_url"]
        text = requests.get(results_url, headers={"x-api-key": api_key, "anthropic-version": "2023-06-01"}).text
        for line in text.splitlines():
            obj = json.loads(line)
            meta = id_map[obj["custom_id"]]
            result = obj["result"]
            usage = None
            if result["type"] == "succeeded":
                content = result["message"]["content"]
                raw = "".join(b["text"] for b in content if b["type"] == "text").strip()
                usage = result["message"].get("usage")
            else:
                raw = f"ERROR: {result.get('type')} {result.get('error', '')}"
            records.append({**meta, "raw_response": raw,
                            "prediction": parse_prediction(raw) if not raw.startswith("ERROR") else 0,
                            **normalize_usage("anthropic", usage)})

    elif provider == "openai":
        status = batch_info.get("status")
        output_file_id = batch_info.get("output_file_id")

        if status != "completed" or not output_file_id:
            err_id = batch_info.get("error_file_id")
            detail = ""
            if err_id:
                detail = requests.get(f"https://api.openai.com/v1/files/{err_id}/content", headers={"Authorization": f"Bearer {api_key}"}).text[:1000]
            raise RuntimeError(
                f"OpenAI batch did not complete (status={status}). "
                f"errors={json.dumps(batch_info.get('errors'))[:300]} detail={detail}")

        text = requests.get(f"https://api.openai.com/v1/files/{output_file_id}/content", headers={"Authorization": f"Bearer {api_key}"}).text
        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            meta = id_map.get(obj["custom_id"], {})
            usage = None
            if obj.get("error"):
                raw = f"ERROR: {obj['error']}"
                pred = 0
            else:
                raw = obj["response"]["body"]["choices"][0]["message"]["content"].strip()
                pred = parse_prediction(raw)
                usage = obj["response"]["body"].get("usage")
            records.append({**meta, "raw_response": raw, "prediction": pred,
                            **normalize_usage("openai", usage)})

    elif provider == "gemini":
        output_uri = batch_info["response"]["responsesFile"]

        # include both /download/ prefix and :download suffix
        resp = requests.get(f"https://generativelanguage.googleapis.com/download/v1beta/{output_uri}:download?alt=media&key={api_key}")

        if not resp.ok:
            print(f"Download failed with status {resp.status_code}: {resp.text}")
        resp.raise_for_status()

        for line in resp.text.splitlines():
            if not line.strip():
                continue
            obj = json.loads(line)
            meta = id_map[obj["key"]]

            if "response" in obj:
                # Safely get candidates, defaulting to an empty list
                cands = obj["response"].get("candidates", [])

                if not cands:
                    # Handle prompt-level safety blocks
                    feedback = obj["response"].get("promptFeedback", {})
                    raw = f"ERROR: Prompt blocked (feedback={feedback})"
                    pred = -1
                else:
                    cand = cands[0]
                    parts = cand.get("content", {}).get("parts", [])
                    raw = "".join(p.get("text", "") for p in parts).strip()
                    if not raw:
                        raw = f"ERROR: empty (finishReason={cand.get('finishReason')})"
                        pred = -1 if "PROHIBITED" in str(cand) else 0
                    else:
                        pred = parse_prediction(raw)
            else:
                raw = f"ERROR: {obj.get('status', 'unknown')}"
                pred = -1 if "PROHIBITED" in raw else 0

            records.append({**meta, "raw_response": raw, "prediction": pred,
                            **normalize_usage("gemini", (obj.get("response") or {}).get("usageMetadata"))})

    return pd.DataFrame(records)

print("submit/poll/parse functions ready")

## 8. Utilities

`cleanup_openai_files` removes orphaned batch uploads. `submit_batch` now deletes them automatically, but run this once to clear anything left over from earlier attempts. **Do not run it while batches are still validating or in progress** if any of them were created before the auto-delete fix.

In [ ]:
def cleanup_openai_files(api_key, dry_run=True):
    r = requests.get("https://api.openai.com/v1/files", headers={"Authorization": f"Bearer {api_key}"})
    r.raise_for_status()
    batch_files = [f for f in r.json().get("data", []) if f.get("purpose") == "batch"]
    total_gb = sum(f.get("bytes", 0) for f in batch_files) / 1e9
    print(f"{len(batch_files)} batch files, {total_gb:.2f} GB")
    if dry_run:
        print("dry_run=True - nothing deleted. Call with dry_run=False to delete.")
        return batch_files
    for f in batch_files:
        requests.delete(f"https://api.openai.com/v1/files/{f['id']}", headers={"Authorization": f"Bearer {api_key}"})
    print(f"deleted {len(batch_files)} files")
    return batch_files


def openai_batch_status(api_keys, batch_dir):
    """Show the real provider-side status of every submitted OpenAI batch."""
    man = load_manifest(batch_dir / "run_manifest.json")
    rows = []
    for key, entry in man.items():
        if entry.get("provider") != "openai" or not entry.get("batch_id"):
            continue
        r = requests.get(f"https://api.openai.com/v1/batches/{entry['batch_id']}", headers={"Authorization": f"Bearer {api_keys['openai']}"})
        info = r.json()
        rows.append({"job": key, "manifest_status": entry.get("status"), "provider_status": info.get("status"), "counts": info.get("request_counts")})
        
    df = pd.DataFrame(rows)
    if len(df):
        print(df["provider_status"].value_counts().to_string())
    return df


def reset_failed(batch_dir, statuses=("parse_failed", "submit_failed")):
    """Clear failed jobs so they are rebuilt and resubmitted on the next run. Jobs marked 'submitted' or 'done' are left alone."""
    mpath = batch_dir / "run_manifest.json"
    man = load_manifest(mpath)
    dead = [k for k, v in man.items() if v.get("status") in statuses]
    for k in dead:
        del man[k]
    save_manifest(man, mpath)
    print(f"cleared {len(dead)} failed jobs; {len(man)} entries remain")
    return dead

print("utilities ready")

## 9. Orchestration (submit-all, then poll, resumable)

- Phase 1 submits every job and records its batch id in `run_manifest.json`.
- Phase 2 polls them together, so the total wait is roughly one batch window rather than one per job.

If the session drops during Phase 2, just re-run the same call. Completed jobs are skipped and nothing is resubmitted.

In [ ]:
import traceback

def _job_key(provider, dataset, seed, variant, eval_type):
    return f"{provider}|{dataset}|rs{seed}|{variant}|{eval_type}"

def _results_csv(batch_dir, provider, dataset, seed, variant, eval_type):
    return batch_dir / provider / f"results_{dataset}_rs{seed}_{variant}_{eval_type}.csv"

def load_manifest(path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return {}

def save_manifest(manifest, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(manifest, f, indent=1)
    tmp.replace(path)


def build_job_list(providers, datasets, seeds, variants, eval_types):
    jobs = []
    for provider in providers:
        for dataset in datasets:
            for seed in seeds:
                for variant in variants:
                    for eval_type in eval_types:
                        jobs.append((provider, dataset, seed, variant, eval_type))
    return jobs

def submit_all(output_base, batch_dir, api_keys, jobs, manifest, manifest_path):
    print(f"=== PHASE 1: submitting {len(jobs)} jobs ===")
    for provider, dataset, seed, variant, eval_type in jobs:
        key = _job_key(provider, dataset, seed, variant, eval_type)
        entry = manifest.get(key, {})

        if entry.get("status") == "done":
            print(f"  SKIP (already done)   {key}")
            continue
        if entry.get("batch_id"):
            print(f"  SKIP (already submitted) {key} -> {entry['batch_id']}")
            continue
        if _results_csv(batch_dir, provider, dataset, seed, variant, eval_type).exists():
            manifest[key] = {"status": "done", "note": "results csv already present"}
            save_manifest(manifest, manifest_path)
            print(f"  SKIP (csv exists)     {key}")
            continue

        try:
            path, idmap_path = generate_batch_file(output_base, batch_dir, provider, dataset, seed, variant, eval_type)
            batch_id = submit_batch(provider, path, api_keys[provider])
            manifest[key] = {
                "status": "submitted", "batch_id": batch_id, "provider": provider,
                "dataset": dataset, "seed": seed, "variant": variant,
                "eval_type": eval_type, "idmap_path": str(idmap_path),
            }
            save_manifest(manifest, manifest_path)
            print(f"  SUBMITTED {key} -> {batch_id}")
            time.sleep(10)
        except Exception as exc:
            manifest[key] = {"status": "submit_failed", "error": str(exc)[:500]}
            save_manifest(manifest, manifest_path)
            print(f"  !! SUBMIT FAILED {key}: {exc}")
    return manifest

def check_status(provider, batch_id, api_key):
    """Returns (is_finished, info_or_None). Never blocks."""
    try:
        if provider == "anthropic":
            r = requests.get(f"https://api.anthropic.com/v1/messages/batches/{batch_id}", headers={"x-api-key": api_key, "anthropic-version": "2023-06-01"})
            r.raise_for_status()
            return (r.json()["processing_status"] == "ended"), r.json()
        if provider == "openai":
            r = requests.get(f"https://api.openai.com/v1/batches/{batch_id}", headers={"Authorization": f"Bearer {api_key}"})
            r.raise_for_status()
            return (r.json()["status"] in ("completed", "failed", "expired", "cancelled")), r.json()
        if provider == "gemini":
            r = requests.get(f"https://generativelanguage.googleapis.com/v1beta/{batch_id}?key={api_key}")
            r.raise_for_status()
            state = r.json().get("metadata", {}).get("state", "")
            return (state in ("BATCH_STATE_SUCCEEDED", "BATCH_STATE_FAILED", "BATCH_STATE_CANCELLED")), r.json()
    except Exception as exc:
        print(f"    (status check error for {batch_id}: {exc})")
        return False, None
    return False, None

def collect_all(batch_dir, api_keys, manifest, manifest_path, interval=120, max_hours=26):
    pending = {k: v for k, v in manifest.items() if v.get("status") == "submitted" and v.get("batch_id")}
    print(f"\n=== PHASE 2: collecting {len(pending)} jobs ===")

    start = time.time()
    while pending and (time.time() - start) < max_hours * 3600:
        for key in list(pending.keys()):
            entry = pending[key]
            finished, info = check_status(entry["provider"], entry["batch_id"], api_keys[entry["provider"]])
            if not finished:
                continue
            try:
                id_map = load_id_map(Path(entry["idmap_path"]))
                df = download_and_parse(entry["provider"], info, api_keys[entry["provider"]], id_map)
                out_csv = _results_csv(batch_dir, entry["provider"], entry["dataset"], entry["seed"], entry["variant"], entry["eval_type"])
                out_csv.parent.mkdir(parents=True, exist_ok=True)
                df.to_csv(out_csv, index=False)
                flush_usage_log(
                    [{"provider": entry["provider"],
                      "model": MODEL_CONFIGS[entry["provider"]]["model"],
                      "dataset": entry["dataset"], "seed": entry["seed"],
                      "variant": entry["variant"], "eval_type": entry["eval_type"],
                      "n_requests": 1,
                      **{c: int(row.get(c, 0)) for c in USAGE_SUMS if c != "n_requests"}}
                     for _, row in df.iterrows()],
                    USAGE_LOG, batch_api=True)
                manifest[key]["status"] = "done"
                manifest[key]["rows"] = len(df)
                save_manifest(manifest, manifest_path)
                print(f"  DONE {key}: {len(df)} rows -> {out_csv.name}")
            except Exception as exc:
                manifest[key]["status"] = "parse_failed"
                manifest[key]["error"] = str(exc)[:500]
                save_manifest(manifest, manifest_path)
                print(f"  !! PARSE FAILED {key}: {exc}")
                traceback.print_exc()
            del pending[key]

        if pending:
            elapsed = (time.time() - start) / 60
            print(f"  ...{len(pending)} still running ({elapsed:.0f} min elapsed), sleeping {interval}s")
            time.sleep(interval)

    if pending:
        print(f"\n!! {len(pending)} jobs still pending after {max_hours}h. "
              f"Re-run this cell later - it will resume from the manifest without resubmitting.")
    return manifest

def run_all_hardened(output_base, batch_dir, api_keys, providers=("anthropic", "openai", "gemini"), datasets=("pleroma", "mastodon"), seeds=(1, 42, 999), variants=("base", "threshold"), eval_types=("local", "shared_global"), interval=120, max_hours=26):
    # guard against providers="openai" (a string) instead of ("openai",) - a missing trailing comma would otherwise iterate over the characters of the string
    if isinstance(providers, str):
        providers = (providers,)
    if isinstance(datasets, str):
        datasets = (datasets,)
    if isinstance(variants, str):
        variants = (variants,)
    if isinstance(eval_types, str):
        eval_types = (eval_types,)
    if isinstance(seeds, int):
        seeds = (seeds,)
    unknown = set(providers) - set(api_keys)
    if unknown:
        raise ValueError(f"Unknown providers: {unknown}. Expected keys from {set(api_keys)}")

    manifest_path = batch_dir / "run_manifest.json"
    manifest = load_manifest(manifest_path)
    jobs = build_job_list(providers, datasets, seeds, variants, eval_types)

    manifest = submit_all(output_base, batch_dir, api_keys, jobs, manifest, manifest_path)
    manifest = collect_all(batch_dir, api_keys, manifest, manifest_path, interval, max_hours)

    counts = {}
    for v in manifest.values():
        counts[v.get("status", "?")] = counts.get(v.get("status", "?"), 0) + 1
    print("\n=== SUMMARY ===")
    for k, v in sorted(counts.items()):
        print(f"  {k}: {v}")
    print(f"manifest: {manifest_path}")
    return manifest

def show_progress(batch_dir):
    """Run any time, in a fresh session, to see where things stand."""
    manifest = load_manifest(batch_dir / "run_manifest.json")
    if not manifest:
        print("no manifest yet (or it is empty)")
        return
    rows = [{"job": k, **{kk: vv for kk, vv in v.items() if kk in ("status", "batch_id", "rows", "error")}}
            for k, v in manifest.items()]
    df = pd.DataFrame(rows)
    print(df["status"].value_counts().to_string())
    return df

print("run_all_hardened + show_progress ready")

### Launch one provider at a time

Uncomment one line. Note the **trailing comma** inside the parentheses - `("openai")` is a string, not a tuple, and would iterate over its characters.

In [ ]:
# Submit and collect every job for all three models in one call.
manifest = run_all_hardened(OUTPUT_BASE, BATCH_DIR, API_KEYS)

In [ ]:
show_progress(BATCH_DIR)

## 10. Scoring and aggregation

The batch results do not carry `ground_truth` (the id_map holds only routing metadata) so it is merged back from the source prediction CSVs.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

SCORE_SEEDS = (1, 42, 999)
SCORE_DATASETS = ("pleroma", "mastodon")
SCORE_VARIANTS = ("base", "threshold")

def _key(s):
    """Stringify join keys so int/str mismatches between the id_map (JSON) and the source CSVs cannot silently drop rows."""
    return s.astype(str).str.strip()

def summarize_predictions(pred_df, name):
    """Same metric set as the main notebook. Rows with prediction == -1 are API refusals and are EXCLUDED from scoring rather than counted as Non-Toxic."""
    scored = pred_df[pred_df["prediction"] != -1]
    n_excluded = len(pred_df) - len(scored)

    y_true = scored["ground_truth"].astype(int)
    y_pred = scored["prediction"].astype(int)

    return {
        "name": name,
        "n": len(scored),
        "n_excluded": n_excluded,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "toxic_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "toxic_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "toxic_f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    }

def print_report(pred_df, title):
    scored = pred_df[pred_df["prediction"] != -1]
    n_excluded = len(pred_df) - len(scored)
    print(title)
    if n_excluded:
        print(f"(excluded {n_excluded} API-refused rows from metrics)")
    print(classification_report(scored["ground_truth"].astype(int), scored["prediction"].astype(int), labels=[0, 1], target_names=["Non-Toxic", "Toxic"], zero_division=0))
    cm = confusion_matrix(scored["ground_truth"].astype(int), scored["prediction"].astype(int), labels=[0, 1])
    print("Confusion matrix [rows=true, cols=pred]:")
    print(pd.DataFrame(cm, index=["True Non-Toxic", "True Toxic"], columns=["Pred Non-Toxic", "Pred Toxic"]))
    return cm

def count_errors(pred_df, tag):
    raw = pred_df["raw_response"].astype(str)
    n_err = int(raw.str.startswith("ERROR").sum())
    n_refused = int((pred_df["prediction"] == -1).sum())
    if n_err:
        print(f"WARNING: {tag}: {n_err}/{len(pred_df)} calls failed. "
              f"{n_refused} were API refusals (excluded from scoring). "
              f"{n_err - n_refused} were other errors (scored as Non-Toxic).")
    return n_err

def load_batch_results(batch_dir, provider, dataset, seed, variant, eval_type, output_base):
    """Load one batch result CSV and merge ground_truth back in from the source prediction CSV (the batch id_map carries only routing metadata)."""
    path = batch_dir / provider / f"results_{dataset}_rs{seed}_{variant}_{eval_type}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing batch results: {path}")
    res = pd.read_csv(path)

    src = load_job_rows(output_base, dataset, seed, eval_type)[
        ["model_instance_id", "conversation_id", "ground_truth", "text"]].copy()

    res["_mi"] = _key(res["model_instance_id"])
    res["_ci"] = _key(res["conversation_id"])
    src["_mi"] = _key(src["model_instance_id"])
    src["_ci"] = _key(src["conversation_id"])

    merged = res.merge(src[["_mi", "_ci", "ground_truth", "text"]], on=["_mi", "_ci"], how="left", validate="one_to_one")
    missing = merged["ground_truth"].isna().sum()

    if missing:
        raise ValueError(
            f"{missing}/{len(merged)} rows in {path.name} have no ground_truth match. "
            f"Check that SOURCE_MODEL_FOLDER points at a complete run for {dataset} rs{seed}.")

    return merged.drop(columns=["_mi", "_ci"])

def score_one(provider, dataset, variant, output_base, batch_dir, seeds=SCORE_SEEDS, model_folder=None):
    """Score one (model, dataset, prompt_variant) across all seeds and write the full CSV set in the main notebook's layout. Returns the aggregate DataFrame."""
    model_name = MODEL_CONFIGS[provider]["model"]
    folder = model_folder or f"{model_name.replace('.', '_')}_results"
    root = output_base / dataset / folder / variant
    root.mkdir(parents=True, exist_ok=True)

    per_seed_summaries = []

    for seed in seeds:
        out_dir = root / f"rs{seed}"
        out_dir.mkdir(parents=True, exist_ok=True)

        local_df = load_batch_results(batch_dir, provider, dataset, seed, variant, "local", output_base)
        shared_df = load_batch_results(batch_dir, provider, dataset, seed, variant, "shared_global", output_base)

        local_df.to_csv(out_dir / "local_predictions.csv", index=False)
        shared_df.to_csv(out_dir / "shared_global_predictions.csv", index=False)

        # per-client summaries
        local_rows, shared_rows = [], []
        for iid in local_df["model_instance_id"].unique():
            sub = local_df[local_df["model_instance_id"] == iid]
            row = summarize_predictions(sub, name=f"local_{iid}"); row["instance_id"] = iid
            local_rows.append(row)

        for iid in shared_df["model_instance_id"].unique():
            sub = shared_df[shared_df["model_instance_id"] == iid]
            row = summarize_predictions(sub, name=f"shared_global_{iid}"); row["instance_id"] = iid
            shared_rows.append(row)

        local_per_client = pd.DataFrame(local_rows)
        shared_per_client = pd.DataFrame(shared_rows)
        local_per_client.to_csv(out_dir / "local_per_client_summary.csv", index=False)
        shared_per_client.to_csv(out_dir / "shared_global_per_client_summary.csv", index=False)

        local_cm = print_report(local_df, f"[{model_name} | {dataset} | {variant} | rs{seed}] Local evaluation")
        shared_cm = print_report(shared_df, f"[{model_name} | {dataset} | {variant} | rs{seed}] Shared global evaluation")
        count_errors(local_df, f"rs{seed} local")
        count_errors(shared_df, f"rs{seed} shared")

        final_summary_df = pd.DataFrame([
            summarize_predictions(local_df, "Local LLMs on local test sets"),
            summarize_predictions(shared_df, "Local LLMs on shared global test set"),
        ])

        final_summary_df.loc[0, "macro_avg_accuracy_across_clients"] = local_per_client["accuracy"].mean()
        final_summary_df.loc[0, "macro_avg_f1_across_clients"] = local_per_client["macro_f1"].mean()
        final_summary_df.loc[1, "macro_avg_accuracy_across_clients"] = shared_per_client["accuracy"].mean()
        final_summary_df.loc[1, "macro_avg_f1_across_clients"] = shared_per_client["macro_f1"].mean()
        final_summary_df.insert(0, "random_state", seed)

        final_summary_df.to_csv(out_dir / "final_summary.csv", index=False)
        pd.DataFrame(local_cm, index=["True Non-Toxic", "True Toxic"], columns=["Pred Non-Toxic", "Pred Toxic"]).to_csv(out_dir / "local_confusion_matrix.csv")
        pd.DataFrame(shared_cm, index=["True Non-Toxic", "True Toxic"], columns=["Pred Non-Toxic", "Pred Toxic"]).to_csv(out_dir / "shared_global_confusion_matrix.csv")

        per_seed_summaries.append(final_summary_df)

    aggregate_df = pd.concat(per_seed_summaries, ignore_index=True)
    aggregate_df.to_csv(root / "aggregate_summary_all_seeds.csv", index=False)

    metric_cols = ["accuracy", "macro_f1", "micro_f1", "weighted_f1",
                   "toxic_precision", "toxic_recall", "toxic_f1",
                   "macro_avg_accuracy_across_clients", "macro_avg_f1_across_clients"]

    mean_std_df = aggregate_df.groupby("name")[metric_cols].agg(["mean", "std"]).round(4)
    mean_std_df.to_csv(root / "aggregate_mean_std_across_seeds.csv")

    print(f"\n>>> {model_name} | {dataset} | {variant}")
    display(mean_std_df)
    print("saved to:", root)
    return aggregate_df, mean_std_df

def score_all(provider, output_base=None, batch_dir=None, datasets=SCORE_DATASETS, variants=SCORE_VARIANTS, seeds=SCORE_SEEDS):
    """Score every (dataset, prompt_variant) combination for one provider/model."""
    output_base = output_base or OUTPUT_BASE
    batch_dir = batch_dir or BATCH_DIR
    out = {}
    for dataset in datasets:
        for variant in variants:
            out[(dataset, variant)] = score_one(provider, dataset, variant, output_base, batch_dir, seeds)
    return out

print("scoring functions ready.")

### Score one provider once all 24 of its jobs report `done`

In [ ]:
# Score every model once its jobs report "done".
# Each call writes per-seed CSVs plus aggregate_summary_all_seeds.csv and
# aggregate_mean_std_across_seeds.csv, per (model, dataset, prompt variant).

res_anthropic = score_all("anthropic") # Claude Opus 5
res_openai = score_all("openai") # GPT-5.6 Sol
res_gemini = score_all("gemini") # Gemini 3.7 Flash

## 11. Token usage per prompt & dataset

Reports the tokens each model consumed, per dataset and prompt variant.

In [ ]:
import pandas as pd
from pathlib import Path

USAGE_COLUMNS = USAGE_KEYS + USAGE_SUMS + ["batch_api"]

def load_usage_log(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"No usage log at {path}. Re-run the experiment cells, or point USAGE_LOG at the log you have.")
    df = pd.read_csv(path)
    missing = [c for c in USAGE_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"usage log {path} is missing columns: {missing}")
    for c in USAGE_SUMS + ["batch_api"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df

def usage_per_prompt_and_dataset(usage_log_path, by=("dataset", "model", "variant")):
    df = load_usage_log(usage_log_path)
    agg = (df.groupby(list(by), as_index=False)
             .agg(n_requests=("n_requests", "sum"),
                  input_tokens=("input_tokens", "sum"),
                  cached_read_tokens=("cached_read_tokens", "sum"),
                  cache_write_tokens=("cache_write_tokens", "sum"),
                  output_tokens=("output_tokens", "sum"),
                  reasoning_tokens=("reasoning_tokens", "sum"),
                  batch_api=("batch_api", "max")))

    agg["uncached_input_tokens"] = (agg["input_tokens"] - agg["cached_read_tokens"] - agg["cache_write_tokens"]).clip(lower=0)
    agg["total_tokens"] = agg["input_tokens"] + agg["output_tokens"]
    n = agg["n_requests"].replace(0, pd.NA)
    agg["input_per_conv"] = agg["input_tokens"] / n
    agg["output_per_conv"] = agg["output_tokens"] / n
    agg["pct_input_cached"] = agg["cached_read_tokens"] / agg["input_tokens"].replace(0, pd.NA) * 100
    agg["pct_output_reasoning"] = agg["reasoning_tokens"] / agg["output_tokens"].replace(0, pd.NA) * 100
    agg["batch_api"] = agg["batch_api"].astype(bool)
    return agg.sort_values(list(by)).reset_index(drop=True)

def print_usage_report(agg):
    show = agg.copy()
    for c in ["input_tokens", "uncached_input_tokens", "cached_read_tokens",
              "output_tokens", "reasoning_tokens", "total_tokens"]:
        show[c] = show[c].map(lambda v: f"{int(v):,}")
    for c in ["input_per_conv", "output_per_conv"]:
        show[c] = show[c].map(lambda v: f"{v:,.0f}" if pd.notna(v) else "-")
    for c in ["pct_input_cached", "pct_output_reasoning"]:
        show[c] = show[c].map(lambda v: f"{v:.1f}%" if pd.notna(v) else "-")

    cols = [c for c in ["dataset", "model", "variant", "batch_api", "n_requests",
                        "input_tokens", "cached_read_tokens", "pct_input_cached",
                        "output_tokens", "reasoning_tokens", "pct_output_reasoning",
                        "total_tokens", "input_per_conv", "output_per_conv"]
            if c in show.columns]
    print(show[cols].to_string(index=False))

    tot = agg[["n_requests", "input_tokens", "cached_read_tokens", "output_tokens", "reasoning_tokens", "total_tokens"]].sum()
    print(f"\nTOTAL: {int(tot['n_requests']):,} classifications | "
          f"input {int(tot['input_tokens']):,} "
          f"(cached {int(tot['cached_read_tokens']):,}) | "
          f"output {int(tot['output_tokens']):,} "
          f"(reasoning {int(tot['reasoning_tokens']):,}) | "
          f"total {int(tot['total_tokens']):,}")

usage_table = usage_per_prompt_and_dataset(USAGE_LOG)
print_usage_report(usage_table)
usage_table.to_csv(Path(USAGE_LOG).with_name("usage_summary.csv"), index=False)